In [ ]:
import cobrabox as cb

import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

In [ ]:
# fpathIn = "../data/synthetic/biorealistic/withvar/"
# # fileName = "var_abg_interictal"
# fileName = "var_abg_moving_to_seizure"

fpathIn = "../data/synthetic/biorealistic/"
fileName = "abg_moving_to_seizure_A"

In [ ]:
df = pd.read_csv(fpathIn + fileName + ".csv.xz", compression="xz")
df.head()

In [ ]:
import json
from pathlib import Path

data_json = json.loads(Path(fpathIn + "info_" + fileName + ".json").read_text())

In [ ]:
current_subject = cb.from_numpy(
    df.values,
    dims=["time", "space"],
    sampling_rate=data_json["fs"],
    groupID=data_json["groupID"],
    condition=data_json["condition"],
    subjectID="sub-001",
    extra=data_json,
)
print(current_subject)

In [ ]:
print(f"Sampling rate: {current_subject.sampling_rate}")
# print(f'Seizure start (sec): {current_subject.extra["Seizure start (sec)"]}')
# print(f'Seizure start (samples): {current_subject.extra["Seizure start (samples)"]}')

In [ ]:
labels_SOZ = current_subject.extra["SOZ"]
# sz_start = current_subject.extra["Seizure start (samples)"]
# sz_end = current_subject.extra["Seizure end (samples)"]

N_channels = current_subject.data.shape[1]
time_vector = list(range(current_subject.data.shape[0]))
fs = int(current_subject.sampling_rate)

print(f"Number of channels: {N_channels}")
print(f"Number of time points: {len(time_vector)}")
print(f"Sampling rate: {fs} Hz")

In [ ]:
plt.figure(figsize=(20, 3))
sns.heatmap(
    current_subject.data.T, cmap="bwr", vmin=-5, vmax=5, cbar_kws={"label": "Feature Value"}
)

In [ ]:
eeg_signal = current_subject.data.T

fig = go.Figure()

offset = 10
for ch in range(N_channels):
    if labels_SOZ[ch]:
        use_color = "red"
        legend_group = "SOZ"
    else:
        use_color = "blue"
        legend_group = "Healthy"

    fig.add_trace(
        go.Scatter(
            x=time_vector[:],
            y=eeg_signal[ch] + ch * offset,
            mode="lines",
            name=f"Ch {ch}",
            line=dict(color=use_color),
            hovertemplate=f"Ch {ch}<br>Time: %{{x:.2f}}s<br>Amplitude: %{{y:.2f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Simulated LFP",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (a.u.)",
    hovermode="x unified",
    # height=600,
    # width=1200
)

fig.show()

In [ ]:
feat = cb.feature.LineLength().apply(current_subject)
print(feat)

Make sure here red are SOZ not random

In [ ]:
# Extract feature data and create labels
feat_data = feat.data.values.flatten()
soz_labels = labels_SOZ.copy()
soz_labels
# Create DataFrame for easier plotting
plot_data = pd.DataFrame(
    {
        "Feature Value": feat_data,
        "Channel Type": ["SOZ" if label else "Healthy" for label in soz_labels],
    }
)

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
sns.violinplot(
    data=plot_data,
    x="Channel Type",
    y="Feature Value",
    hue="Channel Type",
    ax=axes[0],
    palette=["#1f77b4", "#d62728"],
    legend=False,
)
axes[0].set_title(
    "Distribution of Feature Values by Channel Type (Violin Plot)", fontsize=12, fontweight="bold"
)
axes[0].set_ylabel("Feature Value")
axes[0].set_xlabel("")

# KDE plot
sns.kdeplot(
    data=plot_data[plot_data["Channel Type"] == "SOZ"],
    x="Feature Value",
    ax=axes[1],
    label="SOZ",
    color="red",
    fill=True,
    alpha=0.5,
)
sns.kdeplot(
    data=plot_data[plot_data["Channel Type"] == "Healthy"],
    x="Feature Value",
    ax=axes[1],
    label="Healthy",
    color="blue",
    fill=True,
    alpha=0.5,
)
axes[1].set_title("Distribution of Feature Values (KDE Plot)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Feature Value")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.show()

# Print statistics
print("\nDistribution Statistics:")
print(plot_data.groupby("Channel Type")["Feature Value"].describe())

In [ ]:
feature = cb.feature.LineLength()
feature_in_time = (
    cb.feature.SlidingWindow(window_size=fs, step_size=fs // 2)
    | feature
    | cb.feature.ConcatAggregate()
).apply(current_subject)

print("=== ConcatAggregate ===")
print(f"  Shape:   {dict(feature_in_time.data.sizes)}")
print(f"  Dims:    {feature_in_time.data.dims}")
print(f"  History: {feature_in_time.history}")
print()

# Inspect individual windows from ConcatAggregate
n_windows = feature_in_time.data.sizes["window"]
print(f"Number of windows retained: {n_windows}")
print(f"Window 0 values (first channel): {feature_in_time.data.isel(window=0).values[:5]} ...")
print(f"Window 1 values (first channel): {feature_in_time.data.isel(window=1).values[:5]} ...")

In [ ]:
import numpy as np

moving_features = np.array(
    [feature_in_time.data.isel(window=iW).to_numpy[:] for iW in range(n_windows)]
)
plt.figure(figsize=(20, 3))
sns.heatmap(moving_features.T, cmap="plasma", cbar_kws={"label": "Feature Value"})

Is it stupid to compute "second-level" feature on top of the first one? Then I need to use 'window' dimension instead of 'time'

In [ ]:
# second_level_result = cb.feature.AmplitudeVariation().apply(feature_in_time)
# print(second_level_result)

In [ ]:
LEN_TS, N = moving_features.shape[0], moving_features.shape[1]
tvec = np.arange(LEN_TS) * 0.5  # Assuming step_size=25 samples at 50 Hz sampling rate

import plotly.graph_objects as go

fig = go.Figure()

offset = -20
for ch in range(N):
    if labels_SOZ[ch]:
        use_color = "red"
        legend_group = "SOZ"
    else:
        use_color = "blue"
        legend_group = "Healthy"

    fig.add_trace(
        go.Scatter(
            x=tvec[:],
            y=moving_features[:, ch],  # + ch*offset,
            mode="lines",
            name=f"Ch {ch}",
            line=dict(color=use_color),
            hovertemplate=f"Ch {ch}<br>Time: %{{x:.2f}}s<br>Amplitude: %{{y:.2f}}<extra></extra>",
        )
    )

fig.update_layout(
    title="Moving line length feature across channels",
    xaxis_title="Time (s)",
    yaxis_title="Amplitude (a.u.)",
    hovermode="x unified",
    height=600,
    width=1200,
)

fig.show()

In [ ]:
import ruptures as rpt
from sklearn.decomposition import PCA

# feature_matrix is (n_windows, n_channels)
# Keep components explaining 95% of the variance
pca = PCA(n_components=0.95)
latent_features = pca.fit_transform(moving_features)

# algo = rpt.KernelCPD(kernel="rbf").fit(latent_features)
# result = algo.predict(n_bkps=5) # Predict 5 change points

algo_pelt = rpt.Pelt(model="l2").fit(latent_features)
result = algo_pelt.predict(pen=100000)  # Adjust pen to get more/fewer points

# --- 3. Visualization ---
rpt.display(latent_features, result, figsize=(10, 6))
plt.title("Change Points Detected with ruptures")
plt.show()

print("Change points detected at window indices:", result)

If I'd have marks in my info file (those would be in seconds or in samples), how would I map them here?

### Granger causality

Of course, computation of GC based on the whole (unstable) timeseries is stupid, but technically possible

In [ ]:
m = cb.feature.GrangerCausalityMatrix(lag=2).apply(current_subject)
sns.heatmap(m.data.values)
plt.title("Granger Causality Matrix (lag=2), whole recording")

In [ ]:
feature = cb.feature.GrangerCausalityMatrix(lag=1)
concat_result = (
    cb.feature.SlidingWindow(window_size=fs * 10, step_size=fs * 5)
    | feature
    | cb.feature.ConcatAggregate()
).apply(current_subject)

nMW = len(concat_result.data)

Can I plot part of signal corresponding to sliding window 10?

Is some version of progress bar possible?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(23, 5))

sns.heatmap(concat_result.data.values[0], ax=axes[0])
axes[0].set_title("Interictal (window 1)")
sns.heatmap(concat_result.data.values[nMW // 2], ax=axes[1])
axes[1].set_title(f"Preictal (window {nMW // 2})")
sns.heatmap(concat_result.data.values[-1], ax=axes[2])
axes[2].set_title(f"Seizure (window {nMW})")

In [ ]:
feature = cb.feature.GrangerCausalityMatrix(lag=5)
concat_result = (
    cb.feature.SlidingWindow(window_size=fs * 10, step_size=fs * 5)
    | feature
    | cb.feature.ConcatAggregate()
).apply(current_subject)

# this works too:
# pipeline = cb.feature.SlidingWindow(window_size=fs*10, step_size=fs*5) | feature | cb.feature.ConcatAggregate()
# concat_result = pipeline.apply(current_subject)

In [ ]:
nMW = len(concat_result.data)

fig, axes = plt.subplots(1, 3, figsize=(23, 5))

sns.heatmap(concat_result.data.values[0], ax=axes[0])
axes[0].set_title("Interictal (window 1)")
sns.heatmap(concat_result.data.values[nMW // 2], ax=axes[1])
axes[1].set_title(f"Preictal (window {nMW // 2})")
sns.heatmap(concat_result.data.values[-1], ax=axes[2])
axes[2].set_title(f"Seizure (window {nMW})")

I should note that the result is relatively far away from the original matrix. But it's difficult to tell what's correct.

### Phase locking value

In [ ]:
feature = cb.feature.PhaseLockingValueMatrix(coords=range(N_channels))
concat_result = (
    cb.feature.SlidingWindow(window_size=fs * 10, step_size=fs * 5)
    | feature
    | cb.feature.ConcatAggregate()
).apply(current_subject)

# Same thing:
# nMW = len(concat_result.data)
# nMW = concat_result.data.shape[0]
nMW = concat_result.data.sizes["window"]

In [ ]:
print(concat_result)

GC without coordinates computes whole matrix, but PLV complains. I prefer the GC behaviour; anyways, it probably should be the same

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(23, 5))

sns.heatmap(concat_result.data.values[0], ax=axes[0])
axes[0].set_title("Interictal (window 1)")
sns.heatmap(concat_result.data.values[nMW // 2], ax=axes[1])
axes[1].set_title(f"Preictal (window {nMW // 2})")
sns.heatmap(concat_result.data.values[-1], ax=axes[2])
axes[2].set_title(f"Seizure (window {nMW})")